# Euclidean K-2DF Corr Model on Continuous ABP

This notebook trains a biased-ELU mutually exclusive Euclidean K-2DF force CNEEP model on continuous-space WCA active Brownian particles.  Particle centers are converted to a fixed-grid Gaussian density field.  Orientation is intentionally hidden from the network so the learned signal corresponds to the apparent coarse-grained density-field irreversibility rather than particle-level active heat.  The model keeps a force-like local decomposition but lets the force depend on the ordered pair `(x_t, x_{t+dt})`, producing `[F(x_t, x_{t+dt}) + F(x_{t+dt}, x_t)] dot (x_{t+dt} - x_t)`.

Training still uses only the coarse field pairs.  In the density-only version, orientation is not part of the observed state, so exact/GT medium entropy-production comparison is disabled; the remaining predicted maps, Euclidean shell spectra, contact diagnostics, and displacement diagnostics follow the ShellForce notebook.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "ABP", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
from argparse import Namespace
from datetime import datetime
import math

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from tqdm import tqdm

from data.ABP import ABPParams, ContinuousABP, ABPFieldizer
from utils.sampler import CartesianSeqSampler

## 1. Hyperparameters

In [ ]:
opt = Namespace()
opt.model_type = "MultiScaleK_2DF"
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# NEEP objective.  alpha=0 is the most transparent setting for checking
# whether the model sees a forward/reverse asymmetry at all.
opt.alpha = 0.0
opt.beta = 1.0
opt.lam = 0.0
opt.threshold = 0.01

# data/model shape
opt.periodic = True
opt.positional = False
opt.n_components = 1
opt.seq_len = 2

# Mutually exclusive Euclidean K-2DF force settings.  This uses bias and ELU in
# models.NEEP_K_2DF, not the no-bias gain variant.
opt.max_distance = 5
opt.include_k0 = True
opt.k_kernel_geometry = "euclidean"
opt.shell_width = 1.0
opt.shell_offset = 0.0

# training
opt.n_iter = 10000
opt.train_batch_size = 2048
opt.test_batch_size = 256
opt.video_batch_size = 256
opt.lr = 1e-3
opt.wd = 1e-6
opt.input_scalar = 1
opt.loss_scalar = 1
opt.scalar = 1
opt.clip_norm = 1
opt.record_freq = 100
opt.seed = 5
opt.n_layer = 2
opt.n_channel = 48
opt.n_hidden = 2
opt.val_ratio = 0.25

# Fixed-grid ABP simulation style shared with the steady-state sanity notebook.
# Change these four knobs first.  grid_size and dx fix the observation grid;
# changing target_phi changes N, not the box/grid.
target_phi = 0.60
grid_size = 32
dx = 2.0
wca_cutoff_pixels = 2.5

box_L = grid_size * dx
sigma = wca_cutoff_pixels * dx / (2.0 ** (1.0 / 6.0))
N_particles = max(1, int(round(4.0 * target_phi * box_L**2 / (math.pi * sigma**2))))

abp_params = ABPParams(
    N=N_particles,
    L=box_L,
    sigma=sigma,
    epsilon=0.5,
    mobility=1.0,
    force_clip=None,  # keep WCA conservative for stable interaction diagnostics
    force_chunk_size=65536,
    v0=10,
    Dr=1e-6,
    Dt=1e-6,
    dt=1e-4,
    seed=123,
    device=opt.device,
)

fieldizer = ABPFieldizer(
    box_size=abp_params.L,
    grid_size=grid_size,
    particle_diameter=abp_params.sigma,
    mode="gaussian",
    include_orientation=False,
    clip_occupancy=False,
    gaussian_sigma=0.5 * abp_params.sigma,
)
opt.n_components = fieldizer.n_channels
opt.input_shape = (grid_size, grid_size)
compute_gt_ep = bool(fieldizer.include_orientation)

# Simulation controls.  Saved pairs are spaced like the steady-state sanity
# notebook.  GT medium EP is computed only when orientation channels are part of
# the observed field.
n_trajs = 6
n_trajs_test = 2
burn_in = 0
n_steps = 2_000_000
save_interval = 1000
n_steps_test = 2_000_000
sim_batch_size = 2
test_sim_batch_size = 1
test_seed = 456
dt_saved = abp_params.dt * save_interval

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)

result_folder = os.path.join(CNEEP_V2_ROOT, "results")
current_result_folder = os.path.join(
    result_folder, f"CorrABP-K2DF-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
)
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, "model_parameter.pth.tar")
best_checkpoint_path = os.path.join(current_result_folder, "best_model_parameter.pth.tar")

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")
print(f"target_phi={target_phi:.3f}, actual_phi={abp_params.phi:.3f}, N={abp_params.N}")
print(f"ABP Pe={abp_params.Pe:.2f}, epsilon={abp_params.epsilon:.3f}, force_clip={abp_params.force_clip}")
print(f"Grid: {grid_size}x{grid_size}, dx={fieldizer.dx:.4f}, L={abp_params.L:.4f}")
print(f"sigma={abp_params.sigma:.4f} ({abp_params.sigma / fieldizer.dx:.3f} px)")
print(f"WCA rc={abp_params.rc:.4f} ({abp_params.rc / fieldizer.dx:.3f} px)")
print(f"Field channels: Gaussian density only  n_components={opt.n_components}, gaussian_sigma={fieldizer.gaussian_sigma:.4f}")
print(f"dt_saved={dt_saved:.4g}, active displacement per saved pair={abp_params.v0 * dt_saved / fieldizer.dx:.3f} pixels")
print(f"Trajectory counts: train={n_trajs}, test={n_trajs_test}, sim_batch_size={sim_batch_size}")
if compute_gt_ep:
    print("Training uses no explicit EP scale; GT medium EP will be accumulated during simulation.")
else:
    print("Training uses no explicit EP scale; GT medium EP comparison is disabled for density-only fields.")
if abp_params.phi > 1.0:
    print("[WARN] phi > 1.0. This is a very soft-overlap/high-density WCA run, not a hard-disk-like packing.")
print(
    f"K-2DF {opt.k_kernel_geometry} shells: include_k0={opt.include_k0}, "
    f"max_distance={opt.max_distance}, shell_width={opt.shell_width}, shell_offset={opt.shell_offset}"
)

## 2. Simulate ABP trajectories and Gaussian fields

In [ ]:
def fields_to_video(fields):
    # fields: [T, B, C, H, W] -> [B, T, H, W] for density-only C=1.
    fields = fields.float()
    if fields.shape[2] != 1:
        raise ValueError(f"Expected density-only field, got {fields.shape[2]} channels.")
    return fields[:, :, 0].permute(1, 0, 2, 3).contiguous()


def concatenate_simulation_chunks(chunks):
    out = {"params": chunks[0]["params"], "times": chunks[0]["times"]}
    for key in [
        "positions",
        "theta",
        "fields",
        "potential",
        "min_distance",
        "mean_force_norm",
        "exact_active_medium_ep",
        "exact_wca_boundary_ep",
        "exact_medium_ep",
    ]:
        if key in chunks[0]:
            concat_dim = 0 if key.endswith("_ep") else 1
            out[key] = torch.cat([chunk[key] for chunk in chunks], dim=concat_dim)
    return out


def simulate_ensemble_in_batches(total_B, params, base_seed, batch_size, label, total_steps):
    chunks = []
    for start in range(0, total_B, batch_size):
        B = min(batch_size, total_B - start)
        batch_params = ABPParams(**{**params.__dict__, "seed": base_seed + start})
        sim = ContinuousABP(batch_params)
        print(f"[INFO] {label} chunk {start // batch_size + 1}: B={B}, seed={batch_params.seed}")
        chunks.append(
            sim.simulate(
                B=B,
                burn_in=burn_in,
                n_steps=total_steps,
                save_interval=save_interval,
                fieldizer=fieldizer,
                show_progress=True,
                save_exact_medium_ep=compute_gt_ep,
            )
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return concatenate_simulation_chunks(chunks)


print(f"[INFO] Generating TRAIN ABP trajectories B={n_trajs}")
sim_train = ContinuousABP(abp_params)
result_train = simulate_ensemble_in_batches(
    n_trajs,
    abp_params,
    abp_params.seed or 0,
    sim_batch_size,
    "TRAIN",
    n_steps,
)
train_states = fields_to_video(result_train["fields"])

print(f"[INFO] Generating TEST ABP trajectories B={n_trajs_test}")
sim_test = ContinuousABP(ABPParams(**{**abp_params.__dict__, "seed": test_seed}))
result_test = simulate_ensemble_in_batches(
    n_trajs_test,
    ABPParams(**{**abp_params.__dict__, "seed": test_seed}),
    test_seed,
    test_sim_batch_size,
    "TEST",
    n_steps_test,
)
test_states = fields_to_video(result_test["fields"])

print("Train field states:", train_states.shape)
print("Test field states: ", test_states.shape)
print("Final field diagnostics:", fieldizer.diagnostics_dict(result_train["positions"][-1].to(opt.device)))

## 3. Prepare tensors and normalization

In [ ]:
opt.M = train_states.shape[0]
opt.L = train_states.shape[1]
opt.M_test = test_states.shape[0]
opt.L_test = test_states.shape[1]

train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

# Keep full videos on CPU; sampled batches move to opt.device inside the
# training/evaluation loops.
train_video = train_states[:M_train_new].float()
val_video = train_states[M_train_new:].float()
test_video = test_states.float()

mean = torch.mean(train_video, dim=(0, 1, 2, 3), keepdim=True)
std = torch.std(train_video, dim=(0, 1, 2, 3), keepdim=True).clamp_min(1e-6)
transform = lambda x: (x - mean.to(x.device)) * opt.input_scalar / std.to(x.device)

train_U = result_train["potential"].numpy().T
test_U = result_test["potential"].numpy().T
test_min_dist = result_test["min_distance"].numpy().T
test_times = result_test["times"].numpy()
test_exact_medium_ep = result_test["exact_medium_ep"].numpy() if compute_gt_ep else None

print("Train video:", train_video.shape)
print("Val video:  ", val_video.shape)
print("Test video: ", test_video.shape)
print(f"Video tensors stored on {train_video.device}; sampled batches move to {opt.device}.")
print("Density mean:", float(mean.detach().cpu()))
print("Density std: ", float(std.detach().cpu()))
if compute_gt_ep:
    print("Exact medium EP grid:", test_exact_medium_ep.shape)
    print("Per-ensemble exact medium EP rate:", test_exact_medium_ep.mean(axis=1) / dt_saved)
else:
    print("Exact medium EP grid: disabled for density-only fields")
print("Mean WCA U train/test:", float(train_U.mean()), float(test_U.mean()))

## 4. Quick data sanity plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, t in enumerate([0, opt.L // 2, opt.L - 1]):
    im = axes[0, col].imshow(train_video[0, t].detach().cpu().numpy().T, origin="lower", cmap="viridis")
    axes[0, col].set_title(f"train Gaussian field t={t}")
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

axes[1, 0].plot(result_train["times"].numpy(), train_U.T, alpha=0.6)
axes[1, 0].set_title("WCA potential")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("U")

axes[1, 1].plot(result_train["times"].numpy(), result_train["min_distance"].numpy() / abp_params.sigma, alpha=0.6)
axes[1, 1].axhline(1.0, color="k", linestyle="--", lw=1)
axes[1, 1].set_title("min distance / sigma")
axes[1, 1].set_xlabel("time")

axes[1, 2].hist(train_U.reshape(-1), bins=40, alpha=0.8)
axes[1, 2].set_title("WCA U distribution")
axes[1, 2].set_xlabel("U")

plt.tight_layout()
plt.show()

## 5. Build K-2DF force model

In [ ]:
from models.NEEP_K_2DF import MultiScaleK_2DF

if opt.model_type == "MultiScaleK_2DF":
    model = MultiScaleK_2DF(opt).to(opt.device)
else:
    raise ValueError(f"Unknown model_type: {opt.model_type}")

optim = torch.optim.AdamW(model.parameters(), opt.lr, weight_decay=opt.wd)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("K branches:", [branch.k for branch in model.branches])
print("Kernel geometry:", model.k_kernel_geometry)
print("shell bounds in pixels:", model.shell_bounds())
print("shell bounds in sigma units:", [(a * fieldizer.dx / abp_params.sigma, b * fieldizer.dx / abp_params.sigma) for a, b in model.shell_bounds()])

## 6. Train

In [ ]:
train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device
)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False
)

best_val_loss = float("inf")
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None
history_train_loss = []
history_val_loss = []
history_iters = []

plt.ion()
fig, ax = plt.subplots(figsize=(8, 5))
line_train, = ax.plot([], [], label="Train Loss")
line_val, = ax.plot([], [], label="Val Loss")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(True)
display_handle = display(fig, display_id=True)
plt.close(fig)

for it in tqdm(range(1, opt.n_iter + 1)):
    model.train()
    batch = next(train_sampler)
    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.stack(slices, dim=1).float().to(opt.device))

    J_all = model(x) / opt.scalar
    ep_density = J_all.sum(dim=1)

    optim.zero_grad()
    if opt.alpha == 0:
        loss = (-ep_density + (torch.exp(-ep_density) - 1)).mean()
    else:
        loss = (
            -(torch.exp(opt.alpha * ep_density) - 1) / opt.alpha
            + (torch.exp(-(1 + opt.alpha) * ep_density) - 1) / (1 + opt.alpha)
        ).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.stack(vslices, dim=1).float().to(opt.device))
                vJ = model(vx) / opt.scalar
                v_ep_density = vJ.sum(dim=1)
                if opt.alpha == 0:
                    vloss = (-v_ep_density + (torch.exp(-v_ep_density) - 1)).sum().item()
                else:
                    vloss = (
                        -(torch.exp(opt.alpha * v_ep_density) - 1) / opt.alpha
                        + (torch.exp(-(1 + opt.alpha) * v_ep_density) - 1) / (1 + opt.alpha)
                    ).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / max(n_val, 1)
        state = {
            "settings": opt.__dict__,
            "state_dict": model.state_dict(),
            "optimizer": optim.state_dict(),
            "iteration": it,
            "channel_mean": mean.detach().cpu(),
            "channel_std": std.detach().cpu(),
            "abp_params": abp_params.__dict__,
            "fieldizer": {
                "box_size": fieldizer.box_size,
                "grid_size": fieldizer.grid_size,
                "particle_diameter": fieldizer.particle_diameter,
                "mode": fieldizer.mode,
                "include_orientation": fieldizer.include_orientation,
                "clip_occupancy": fieldizer.clip_occupancy,
                "gaussian_sigma": fieldizer.gaussian_sigma,
                "gaussian_sigma_pixels": fieldizer.gaussian_sigma_pixels,
                "gaussian_truncate": fieldizer.gaussian_truncate,
                "gaussian_normalize": fieldizer.gaussian_normalize,
            },
            "simulation_style": {
                "target_phi": target_phi,
                "actual_phi": abp_params.phi,
                "grid_size": grid_size,
                "dx": dx,
                "wca_cutoff_pixels": wca_cutoff_pixels,
                "n_particles": abp_params.N,
                "save_exact_medium_ep": compute_gt_ep,
            },
            "dt_saved": dt_saved,
        }
        torch.save(state, current_checkpoint_path)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        smooth_train_loss = loss.item() if smooth_train_loss is None else smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
        smooth_val_loss = avg_val if smooth_val_loss is None else smoothing * smooth_val_loss + (1 - smoothing) * avg_val
        history_train_loss.append(smooth_train_loss)
        history_val_loss.append(smooth_val_loss)
        history_iters.append(it)
        line_train.set_data(history_iters, history_train_loss)
        line_val.set_data(history_iters, history_val_loss)
        ax.relim()
        ax.autoscale_view()
        display_handle.update(fig)

print("Training finished.")
print(f"Best checkpoint: {best_checkpoint_path}")

## 7. Load best or final model

In [ ]:
load_best = True
checkpoint_path = best_checkpoint_path if load_best and os.path.exists(best_checkpoint_path) else current_checkpoint_path
checkpoint = torch.load(checkpoint_path, map_location=opt.device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Loaded {checkpoint_path}")
print(f"Iteration: {checkpoint.get('iteration', 'unknown')}")

## 8. Predict test-pair EP-like increments

In [ ]:
model.eval()
pred_increment = []
pred_shell_increment = []
pair_b = []
pair_t = []

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False
)

with torch.no_grad():
    for batch in tqdm(test_sampler):
        b0 = batch[0].to(test_video.device)
        t0 = batch[1][0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.stack(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar
        shell_inc = J_all.detach().cpu().numpy()
        total_inc = shell_inc.sum(axis=1)
        pred_shell_increment.append(shell_inc)
        pred_increment.append(total_inc)
        pair_b.append(batch[0].detach().cpu().numpy())
        pair_t.append(batch[1][0].detach().cpu().numpy())

pred_increment = np.concatenate(pred_increment)
pred_shell_increment = np.concatenate(pred_shell_increment, axis=0)
pair_b = np.concatenate(pair_b)
pair_t = np.concatenate(pair_t)

min_r = test_min_dist[pair_b, pair_t]
pred_rate = pred_increment / dt_saved
shell_rate = pred_shell_increment / dt_saved
pair_time = test_times[pair_t]

def corrcoef(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

print("Raw predicted pair summary by ensemble")
for ens in range(opt.M_test):
    mask = pair_b == ens
    if not mask.any():
        print(f"  ens {ens}: no predicted pairs")
        continue
    print(
        f"  ens {ens}: pred_mean_rate={pred_rate[mask].mean():.6e}, "
        f"corr(pred, min_dist)={corrcoef(pred_rate[mask], min_r[mask]):.4f}"
    )

## 9. Per-ensemble predicted EP diagnostics

In [ ]:
pred_inc_grid = np.full((opt.M_test, opt.L_test - 1), np.nan, dtype=np.float64)
for inc, b, t in zip(pred_increment, pair_b, pair_t):
    if 0 <= b < opt.M_test and 0 <= t < opt.L_test - 1:
        pred_inc_grid[b, t] = inc

missing = np.isnan(pred_inc_grid)
if missing.any():
    print(f"[WARN] Missing predicted increments in {missing.sum()} pair slots; filling them with 0 for cumulative plots.")
pred_inc_grid = np.nan_to_num(pred_inc_grid, nan=0.0)
pred_rate_grid = pred_inc_grid / dt_saved
time_pairs = test_times[1:]

def finite_pair_mask(*arrays):
    mask = np.ones_like(np.asarray(arrays[0], dtype=float), dtype=bool)
    for arr in arrays:
        mask &= np.isfinite(arr)
    return mask

def raw_r2_score(y_true, y_pred):
    mask = finite_pair_mask(y_true, y_pred)
    if mask.sum() < 3:
        return np.nan
    y = y_true[mask]
    yp = y_pred[mask]
    denom = float(np.sum((y - y.mean()) ** 2))
    if denom <= 1e-30:
        return np.nan
    return float(1.0 - np.sum((y - yp) ** 2) / denom)

exact_medium_inc_grid = None
exact_medium_rate_grid = None
exact_medium_increment = None
exact_medium_rate = None
if compute_gt_ep:
    # Exact ABP medium EP on the same test pairs used for model prediction.
    # These arrays were accumulated during simulation.  The active-work term is
    # summed over every integration step inside each saved interval, so this remains
    # exact even when save_interval > 1.
    required_ep_keys = ["exact_medium_ep"]
    missing_ep_keys = [key for key in required_ep_keys if key not in result_test]
    if missing_ep_keys:
        raise KeyError(f"Missing exact EP arrays from result_test: {missing_ep_keys}")
    exact_medium_inc_grid = result_test["exact_medium_ep"].numpy()
    exact_medium_rate_grid = exact_medium_inc_grid / dt_saved
    exact_medium_increment = exact_medium_inc_grid[pair_b, pair_t]
    exact_medium_rate = exact_medium_increment / dt_saved

print("Per-ensemble predicted EP summary")
for ens in range(opt.M_test):
    mask = pair_b == ens
    if not mask.any():
        print(f"  ens {ens}: no predicted pairs")
        continue
    msg = f"  ens {ens}: pred_mean_rate={pred_rate_grid[ens].mean():.6e}"
    if compute_gt_ep:
        msg += (
            f", exact_mean_rate={exact_medium_rate_grid[ens].mean():.6e}, "
            f"corr={corrcoef(pred_rate[mask], exact_medium_rate[mask]):.4f}, "
            f"raw_R2={raw_r2_score(exact_medium_increment[mask], pred_increment[mask]):.4f}"
        )
    print(msg)

if not compute_gt_ep:
    print("GT exact medium EP comparison is disabled because orientation is not in the observed field.")

n_ens = opt.M_test
fig, axes = plt.subplots(n_ens, 2, figsize=(14, max(3.4 * n_ens, 4)), squeeze=False)
for row, ens in enumerate(range(n_ens)):
    if compute_gt_ep:
        axes[row, 0].plot(time_pairs, exact_medium_rate_grid[ens], label="exact medium", lw=1.8)
    axes[row, 0].plot(time_pairs, pred_rate_grid[ens], label="predicted raw", lw=1.2, alpha=0.75)
    axes[row, 0].set_ylabel("EP rate")
    axes[row, 0].set_title(f"ensemble {ens}: predicted rate")
    axes[row, 0].legend()

    if compute_gt_ep:
        axes[row, 1].plot(time_pairs, np.cumsum(exact_medium_inc_grid[ens]), label="exact medium", lw=1.8)
    axes[row, 1].plot(time_pairs, np.cumsum(pred_inc_grid[ens]), label="predicted raw", lw=1.2, alpha=0.75)
    axes[row, 1].set_ylabel("cumulative increment")
    axes[row, 1].set_title(f"ensemble {ens}: cumulative predicted EP")
    axes[row, 1].legend()

axes[-1, 0].set_xlabel("time")
axes[-1, 1].set_xlabel("time")

plt.tight_layout()
figure_name = "abp_exact_medium_ep_raw_per_ensemble.png" if compute_gt_ep else "abp_predicted_ep_per_ensemble.png"
plt.savefig(os.path.join(current_result_folder, figure_name), dpi=150)
plt.show()

## 10. Per-ensemble predicted EP and contact diagnostics

In [ ]:
def smooth(x, window=11):
    window = min(window, len(x))
    if window <= 1:
        return x
    return np.convolve(x, np.ones(window) / window, mode="same")

fig, axes = plt.subplots(opt.M_test, 2, figsize=(14, max(3.4 * opt.M_test, 4)), squeeze=False)
for ens in range(opt.M_test):
    axes[ens, 0].plot(time_pairs, pred_rate_grid[ens], alpha=0.35, label="predicted raw")
    axes[ens, 0].plot(time_pairs, smooth(pred_rate_grid[ens]), lw=2, label="pred smooth")
    if compute_gt_ep:
        axes[ens, 0].plot(time_pairs, smooth(exact_medium_rate_grid[ens]), lw=2, label="exact smooth")
    axes[ens, 0].set_ylabel("EP rate")
    axes[ens, 0].set_title(f"ensemble {ens}: predicted EP rate")
    axes[ens, 0].legend()

    minr_series = test_min_dist[ens, :-1] / abp_params.sigma
    axes[ens, 1].plot(time_pairs, minr_series, alpha=0.35, color="tab:red")
    axes[ens, 1].plot(time_pairs, smooth(minr_series), lw=2, color="tab:red")
    axes[ens, 1].axhline(1.0, color="k", linestyle="--", lw=1)
    axes[ens, 1].set_ylabel("min r/sigma")
    axes[ens, 1].set_title(f"ensemble {ens}: contact diagnostic")

axes[-1, 0].set_xlabel("time")
axes[-1, 1].set_xlabel("time")

plt.tight_layout()
figure_name = "abp_raw_ep_contact_by_ensemble.png" if compute_gt_ep else "abp_predicted_ep_contact_by_ensemble.png"
plt.savefig(os.path.join(current_result_folder, figure_name), dpi=150)
plt.show()

## 11. Local predicted EP map and video

In [ ]:
from matplotlib.animation import FuncAnimation


def predicted_local_ep_map(ens, frame):
    x = transform(
        torch.stack(
            [test_video[ens:ens+1, frame], test_video[ens:ens+1, frame + 1]],
            dim=1,
        ).float().to(opt.device)
    )
    with torch.no_grad():
        maps = model(x, return_maps=True) / opt.scalar
    return maps[0].sum(dim=0).detach().cpu().numpy() / dt_saved


def density_frame_array(ens, frame):
    frame_tensor = test_video[ens, frame]
    if frame_tensor.ndim == 3:
        frame_tensor = frame_tensor[0]
    return frame_tensor.detach().cpu().numpy()


ens = 0
frame = int(0.6 * (opt.L_test - 1))
density_frame = density_frame_array(ens, frame)
pred_map_rate = predicted_local_ep_map(ens, frame)

v = np.nanpercentile(np.abs(pred_map_rate), 99)
v = max(v, 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(density_frame.T, origin="lower", cmap="viridis")
axes[0].set_title(f"Gaussian density, ens={ens}, t={test_times[frame]:.3f}")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(pred_map_rate.T, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[1].set_title("predicted local EP rate")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
static_path = os.path.join(current_result_folder, "abp_local_predicted_ep_map.png")
plt.savefig(static_path, dpi=150)
plt.show()
print("Saved local EP map:", static_path)

video_frame_ids = np.linspace(0, opt.L_test - 2, min(120, opt.L_test - 1), dtype=int)
density_video = []
pred_video = []
for t in tqdm(video_frame_ids, desc="local EP video"):
    density_video.append(density_frame_array(ens, int(t)))
    pred_video.append(predicted_local_ep_map(ens, int(t)))

density_video = np.asarray(density_video)
pred_video = np.asarray(pred_video)
video_vmax = max(1e-12, float(np.nanpercentile(np.abs(pred_video), 99)))
density_vmax = max(1e-12, float(np.nanpercentile(density_video, 99.5)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dens_im = axes[0].imshow(density_video[0].T, origin="lower", cmap="viridis", vmin=0, vmax=density_vmax)
pred_im = axes[1].imshow(pred_video[0].T, origin="lower", cmap="RdBu_r", vmin=-video_vmax, vmax=video_vmax)
axes[0].set_title("Gaussian density")
axes[1].set_title("predicted local EP rate")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.colorbar(dens_im, ax=axes[0], fraction=0.046)
plt.colorbar(pred_im, ax=axes[1], fraction=0.046)
title = fig.suptitle("")

def update_local_video(k):
    frame_id = int(video_frame_ids[k])
    dens_im.set_data(density_video[k].T)
    pred_im.set_data(pred_video[k].T)
    title.set_text(f"ens={ens}, frame={frame_id}/{opt.L_test - 2}, t={test_times[frame_id]:.3f}")
    return dens_im, pred_im, title

anim = FuncAnimation(fig, update_local_video, frames=len(video_frame_ids), interval=90, blit=False)
html = anim.to_jshtml()
video_html_path = os.path.join(current_result_folder, "abp_local_predicted_ep_video.html")
with open(video_html_path, "w", encoding="utf-8") as f:
    f.write(html)
plt.close(fig)
print("Saved local EP video HTML:", video_html_path)
display(HTML(html))

## 12. Euclidean K-shell spectrum

In [ ]:
mean_shell = shell_rate.mean(axis=0)
stderr_shell = shell_rate.std(axis=0) / np.sqrt(max(len(shell_rate), 1))
bounds_px = model.shell_bounds()
centers_sigma = np.array([0.0 if hi == 0 else 0.5 * (lo + hi) * fieldizer.dx / abp_params.sigma for lo, hi in bounds_px])
widths_sigma = np.array([0.75 * fieldizer.dx / abp_params.sigma if hi == 0 else max((hi - lo) * fieldizer.dx / abp_params.sigma, 0.1) for lo, hi in bounds_px])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(centers_sigma, mean_shell, yerr=stderr_shell, width=0.8 * widths_sigma, capsize=3, alpha=0.8)
axes[0].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[0].set_xlabel("Euclidean annulus radius / sigma")
axes[0].set_ylabel("mean predicted Euclidean K-shell rate")
axes[0].set_title("Euclidean K-2DF spectrum")
axes[0].legend()

axes[1].plot(centers_sigma, np.cumsum(mean_shell), "o-")
axes[1].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[1].set_xlabel("Euclidean annulus radius / sigma")
axes[1].set_ylabel("cumulative predicted rate")
axes[1].set_title("Cumulative Euclidean K-shell contribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_k2df_spectrum.png"), dpi=150)
plt.show()

for idx, (lo, hi) in enumerate(bounds_px):
    print(
        f"branch {idx:02d}: pixels=({lo:.2f}, {hi:.2f}], "
        f"sigma_units=({lo * fieldizer.dx / abp_params.sigma:.3f}, {hi * fieldizer.dx / abp_params.sigma:.3f}], "
        f"mean_rate={mean_shell[idx]:.6e}"
    )

## 13. Particle displacement distribution vs Euclidean K-shell spectrum

In [ ]:
# This diagnostic uses hidden particle orientations from the simulator for
# interpretation only.  The density-only model did not receive orientation
# channels as input.
pos = result_test["positions"].numpy()  # [T, B, N, 2]
theta = result_test["theta"].numpy()    # [T, B, N]

# User controls for this diagnostic only.
# diagnostic_stride is measured in saved frames, so the physical interval is
# diagnostic_stride * dt_saved.
diagnostic_stride = 1

if diagnostic_stride < 1:
    raise ValueError("diagnostic_stride must be >= 1.")
if diagnostic_stride >= pos.shape[0]:
    raise ValueError("diagnostic_stride is longer than the saved trajectory.")

pos0 = pos[:-diagnostic_stride]
pos1 = pos[diagnostic_stride:]
theta0 = theta[:-diagnostic_stride]

dr = pos1 - pos0
dr = dr - abp_params.L * np.round(dr / abp_params.L)
direction = np.stack([np.cos(theta0), np.sin(theta0)], axis=-1)
perp_direction = np.stack([-np.sin(theta0), np.cos(theta0)], axis=-1)

disp = np.linalg.norm(dr, axis=-1)
parallel = np.sum(dr * direction, axis=-1)
perp = np.sum(dr * perp_direction, axis=-1)

flat_disp = disp.reshape(-1)
flat_parallel = parallel.reshape(-1)
flat_perp = perp.reshape(-1)

diagnostic_dt = diagnostic_stride * dt_saved
free_active_step = abp_params.v0 * diagnostic_dt
trans_noise_rms = np.sqrt(4.0 * abp_params.Dt * diagnostic_dt)
pixel_size = fieldizer.dx

flat_disp_px = flat_disp / pixel_size
flat_parallel_px = flat_parallel / pixel_size
flat_perp_px = flat_perp / pixel_size
free_active_step_px = free_active_step / pixel_size
trans_noise_rms_px = trans_noise_rms / pixel_size

print(f"diagnostic_stride:      {diagnostic_stride} saved frames")
print(f"diagnostic_dt:          {diagnostic_dt:.6e}")
print(f"pixel_size dx:          {pixel_size:.6e}")
print(f"mean |dr| [px]:         {flat_disp_px.mean():.6e}")
print(f"mean e(theta).dr [px]:  {flat_parallel_px.mean():.6e}")
print(f"std  e(theta).dr [px]:  {flat_parallel_px.std():.6e}")
print(f"mean e_perp.dr [px]:    {flat_perp_px.mean():.6e}")
print(f"free active step [px]:  {free_active_step_px:.6e}")
print(f"2D Brownian rms [px]:   {trans_noise_rms_px:.6e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(flat_parallel_px, bins=80, density=True, alpha=0.75)
axes[0].axvline(free_active_step_px, color="k", linestyle="--", label="free active step")
axes[0].axvline(flat_parallel_px.mean(), color="tab:red", lw=2, label="mean")
axes[0].set_xlabel("e(theta) dot dr [pixels]")
axes[0].set_ylabel("density")
axes[0].set_title("Forward displacement")
axes[0].legend()

axes[1].hist(flat_perp_px, bins=80, density=True, alpha=0.75, color="tab:purple")
axes[1].axvline(0, color="k", linestyle="--")
axes[1].axvline(flat_perp_px.mean(), color="tab:red", lw=2, label="mean")
axes[1].set_xlabel("e_perp(theta) dot dr [pixels]")
axes[1].set_title("Transverse displacement")
axes[1].legend()

axes[2].hist(flat_disp_px, bins=80, density=True, alpha=0.75, color="tab:green")
axes[2].axvline(free_active_step_px, color="k", linestyle="--", label="free active step")
axes[2].set_xlabel("|dr| [pixels]")
axes[2].set_title("Step length")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_displacement_distribution.png"), dpi=150)
plt.show()
print("Detailed self-step bin statistics are omitted; the histogram above is the displacement diagnostic.")

## 14. Inspect Euclidean K-shell masks

In [ ]:
branches = [branch for branch in model.branches if branch.k > 0]
n_show = min(4, len(branches))
fig, axes = plt.subplots(1, n_show, figsize=(3.3 * n_show, 3.2))
if n_show == 1:
    axes = [axes]
for ax, branch in zip(axes, branches[:n_show]):
    offsets = branch.masked_conv.offsets.detach().cpu().numpy()
    ax.scatter(offsets[:, 1], offsets[:, 0], s=45)
    circle_outer = plt.Circle((0, 0), branch.r_outer, fill=False, color="k", linestyle="--")
    circle_inner = plt.Circle((0, 0), branch.r_inner, fill=False, color="gray", linestyle=":")
    ax.add_patch(circle_outer)
    ax.add_patch(circle_inner)
    ax.axhline(0, color="0.8", lw=1)
    ax.axvline(0, color="0.8", lw=1)
    ax.set_aspect("equal")
    ax.set_title(f"k={branch.k}: ({branch.r_inner:.1f},{branch.r_outer:.1f}] px")
plt.tight_layout()
plt.show()